<div style="background: linear-gradient(135deg, #1d4ed8, #3b82f6); padding: 2rem; border-radius: 16px; color: white; text-align: center;">
  <h1 style="font-size: 2.5rem; margin: 0;">🚗 Drive Wise</h1>
  <p style="font-size: 1.2rem; margin: 0.5rem 0 0 0; opacity: 0.9;">Metadata-Aware Automotive RAG Assistant</p>
  <p style="font-size: 0.9rem; margin: 0.5rem 0 0 0; opacity: 0.75;">Powered by Google Gemini 2.5 Flash · Hybrid Semantic + Keyword Retrieval</p>
</div>

---

## 📋 What This Notebook Does

This is a **complete, self-contained demo** of the DriveWise RAG pipeline. Just run all cells and use the interactive chat at the bottom!

### 🚗 Cars Available
| Brand | Models |
|-------|--------|
| Honda | Amaze |
| Hyundai | Aura, Grand I10 Nios, Verna |
| Mahindra | Scorpio N, Thar, XUV700 |
| Tata | Sierra |
| Toyota | Fortuner, Innova Hycross |

> **⚡ Quick Start**: Click **Runtime → Run All**. Add your `GOOGLE_API_KEY` in the 🔑 Secrets panel (left sidebar). Then scroll to the bottom to use the interactive chat!

## Step 1 — Install Dependencies

In [ ]:
!pip install -q google-generativeai pypdf numpy ipywidgets
print("✅ All packages installed!")

## Step 2 — Configure Your Gemini API Key

Get a **free** key from [aistudio.google.com/apikey](https://aistudio.google.com/apikey)  
**On Colab**: Use the 🔑 Secrets panel (left sidebar) → add `GOOGLE_API_KEY`

In [ ]:
import os
import json
import time
import textwrap
import numpy as np
import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── API Key ────────────────────────────────────────────────────────────────────
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
    print("🔑 API key loaded from Colab Secrets.")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('GOOGLE_API_KEY')
    if api_key:
        print("🔑 API key loaded from environment variable.")

if not api_key:
    api_key = "YOUR_GEMINI_API_KEY_HERE"  # ← Paste your key here if needed
    print("⚠️  No secret found — using placeholder. Please set GOOGLE_API_KEY in Secrets.")

genai.configure(api_key=api_key)
print("✅ Gemini API configured!")

## Step 3 — Load the Brochure Index

In [ ]:
import urllib.request

INDEX_URL  = "https://raw.githubusercontent.com/avanishar/drivewiseapp/main/index/brochure_index.json"
INDEX_PATH = "brochure_index.json"

if not os.path.exists(INDEX_PATH):
    print("⬇️  Downloading brochure index (~13MB)...")
    try:
        urllib.request.urlretrieve(INDEX_URL, INDEX_PATH)
        print(f"✅ Downloaded ({os.path.getsize(INDEX_PATH)/1024/1024:.1f} MB)")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        print("📌 Manually upload brochure_index.json from the project's index/ folder.")
else:
    print(f"✅ Index already present ({os.path.getsize(INDEX_PATH)/1024/1024:.1f} MB)")

print("⏳ Loading into memory...")
with open(INDEX_PATH, 'r', encoding='utf-8') as f:
    index_data = json.load(f)

for chunk in index_data.get('chunks', []):
    if 'embedding' in chunk and chunk['embedding'] is not None:
        chunk['embedding'] = np.array(chunk['embedding'], dtype=np.float32)

files  = index_data.get('files', {})
chunks = index_data.get('chunks', [])
print(f"✅ Loaded: {len(chunks)} chunks from {len(files)} brochures")
print()
for fname, meta in files.items():
    print(f"  🚗 {meta.get('brand')} {meta.get('model'):25s} → {meta.get('chunks_count')} chunks")

## Step 4 — RAG Engine & Generator

In [ ]:
STOPWORDS = {
    "a","about","above","after","again","all","am","an","and","any","are","as","at",
    "be","because","been","before","being","below","between","both","but","by","can",
    "did","do","does","doing","down","during","each","for","from","had","has","have",
    "having","he","her","here","him","his","how","i","if","in","into","is","it","its",
    "me","more","most","my","no","nor","not","of","off","on","once","only","or","other",
    "our","out","over","own","same","she","should","so","some","such","than","that",
    "the","their","them","then","there","these","they","this","those","through","to",
    "too","under","until","up","very","was","we","were","what","when","where","which",
    "while","who","whom","why","with","you","your"
}

def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return float(dot / (n1 * n2)) if n1 > 0 and n2 > 0 else 0.0

def keyword_score(query, text):
    words = [w.strip("?,.:;!\"'()").lower() for w in query.split()]
    words = [w for w in words if w and w not in STOPWORDS]
    if not words:
        return 0.0
    text_lower = text.lower()
    return sum(1 for w in words if w in text_lower) / len(words)

def retrieve_chunks(query, brand, model, limit=4):
    all_chunks = index_data.get('chunks', [])
    filtered = [
        c for c in all_chunks
        if c.get('brand','').lower() == brand.lower()
        and c.get('model','').lower() == model.lower()
    ]
    if not filtered:
        return []
    try:
        emb_resp = genai.embed_content(model='models/gemini-embedding-001', content=query)
        query_emb = emb_resp['embedding']
    except Exception:
        return []
    scored = []
    for c in filtered:
        emb = c.get('embedding')
        if emb is None:
            continue
        sem = cosine_similarity(query_emb, emb)
        kw  = keyword_score(query, c['text'])
        scored.append({**c, 'score': 0.8*sem + 0.2*kw, 'sem': sem, 'kw': kw})
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:limit]

def generate_answer(query, brand, model):
    chunks = retrieve_chunks(query, brand, model)
    if not chunks:
        return f"❌ No brochure data found for '{brand} {model}'. Please check the car name.", []
    context_str = ""
    for idx, c in enumerate(chunks):
        context_str += f"\n--- Source [{idx+1}] (Page {c['page']}, Section: {c['section']}) ---\n{c['text']}\n"
    system_instruction = f"""You are an expert automotive assistant for Drive Wise.
Answer the user's query about the {brand} {model} strictly from the brochure excerpts provided.
Rules:
1. Use ONLY the provided brochure context. Do not use outside knowledge.
2. If information is missing, say: "This information is not available in the brochure."
3. Use inline citations like [1], [2] matching the source numbers.
4. Be clear, structured, and easy for a car buyer to understand."""
    prompt = f"""Brochure Context for {brand} {model}:
{context_str}
User Query: \"{query}\"
Grounded Answer:"""
    try:
        gen_model = genai.GenerativeModel(
            model_name="models/gemini-2.5-flash",
            system_instruction=system_instruction
        )
        response = gen_model.generate_content(
            prompt, generation_config=genai.types.GenerationConfig(temperature=0.1)
        )
        return response.text.strip(), chunks
    except Exception as e:
        return f"❌ Error: {e}", []

# Build car map from index
CAR_MAP = {}
for fname, meta in index_data.get('files', {}).items():
    b = meta.get('brand', 'Unknown')
    m = meta.get('model', 'Unknown')
    if b not in CAR_MAP:
        CAR_MAP[b] = []
    if m not in CAR_MAP[b]:
        CAR_MAP[b].append(m)

print("✅ RAG engine ready!")
print(f"   Available brands: {sorted(CAR_MAP.keys())}")

---
## 🤖 Pre-Set Demo Queries
Run the cells below to see example answers across different cars.

In [ ]:
# Demo 1 — Honda Amaze Safety
answer, sources = generate_answer("What safety features does the Honda Amaze have?", "Honda", "Amaze")
print("━"*65)
print("🚗 Honda Amaze | Safety Features")
print("━"*65)
print(answer)
print(f"\n📚 Sources used: {len(sources)} chunks")
for s in sources:
    print(f"   [{sources.index(s)+1}] Page {s['page']} · {s['section']} · Score: {s['score']:.3f}")

In [ ]:
# Demo 2 — Mahindra Thar Engine
answer, sources = generate_answer("What are the engine options and power output of the Mahindra Thar?", "Mahindra", "Thar")
print("━"*65)
print("🚗 Mahindra Thar | Engine & Performance")
print("━"*65)
print(answer)
print(f"\n📚 Sources used: {len(sources)} chunks")
for s in sources:
    print(f"   [{sources.index(s)+1}] Page {s['page']} · {s['section']} · Score: {s['score']:.3f}")

In [ ]:
# Demo 3 — Tata Sierra Infotainment
answer, sources = generate_answer("What infotainment and connected car features does the Tata Sierra offer?", "Tata", "Sierra")
print("━"*65)
print("🚗 Tata Sierra | Infotainment & Tech")
print("━"*65)
print(answer)
print(f"\n📚 Sources used: {len(sources)} chunks")
for s in sources:
    print(f"   [{sources.index(s)+1}] Page {s['page']} · {s['section']} · Score: {s['score']:.3f}")

---
# 🎮 Interactive Chat — Ask Your Own Questions!

**Run the cell below** to launch the interactive DriveWise chatbot.
- Select a **Brand** and **Model** from the dropdowns
- Type any question in the text box
- Click **Ask DriveWise** to get an answer with source citations

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          🚗 Drive Wise Interactive Chat Widget              ║
# ╚══════════════════════════════════════════════════════════════╝

chat_history = []  # stores all Q&A pairs

# ── Widgets ────────────────────────────────────────────────────────────────────
brand_options = sorted(CAR_MAP.keys())

brand_dropdown = widgets.Dropdown(
    options=brand_options,
    value=brand_options[0],
    description='🏷️ Brand:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='240px')
)

model_dropdown = widgets.Dropdown(
    options=CAR_MAP[brand_options[0]],
    description='🚗 Model:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='240px')
)

query_input = widgets.Textarea(
    placeholder='Type your question here...\nE.g. "What is the mileage?" or "How many airbags does it have?"',
    description='',
    layout=widgets.Layout(width='100%', height='80px')
)

ask_button = widgets.Button(
    description=' Ask DriveWise',
    button_style='primary',
    icon='car',
    layout=widgets.Layout(width='180px', height='40px')
)

clear_button = widgets.Button(
    description=' Clear Chat',
    button_style='warning',
    icon='trash',
    layout=widgets.Layout(width='140px', height='40px')
)

status_label = widgets.HTML(value='<span style="color:#64748b; font-size:0.9rem;">Ready to answer your questions!</span>')

chat_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #e2e8f0',
        border_radius='12px',
        padding='16px',
        min_height='200px',
        max_height='600px',
        overflow_y='auto',
        background_color='#f8fafc'
    )
)

# ── Update model dropdown when brand changes ────────────────────────────────────
def on_brand_change(change):
    model_dropdown.options = CAR_MAP.get(change['new'], [])

brand_dropdown.observe(on_brand_change, names='value')

# ── Render full chat history ────────────────────────────────────────────────────
def render_chat():
    with chat_output:
        clear_output(wait=True)
        if not chat_history:
            display(HTML("""
            <div style="text-align:center; padding:40px; color:#94a3b8;">
              <div style="font-size:3rem;">🚗</div>
              <div style="font-size:1.1rem; margin-top:8px;">Ask anything about the selected car!</div>
              <div style="font-size:0.85rem; margin-top:4px;">Questions about mileage, safety, features, dimensions, engine...</div>
            </div>
            """))
            return
        for item in chat_history:
            # User bubble
            display(HTML(f"""
            <div style="margin:12px 0; display:flex; justify-content:flex-end;">
              <div style="background:#eff6ff; border:1px solid #bfdbfe; border-radius:16px 16px 4px 16px;
                          padding:12px 16px; max-width:80%; font-size:0.95rem; color:#1e3a5f;">
                <b>👤 You:</b><br>{item['query']}
              </div>
            </div>
            """))
            # Answer bubble
            answer_html = item['answer'].replace('\n', '<br>')
            display(HTML(f"""
            <div style="margin:8px 0 16px 0;">
              <div style="background:#ffffff; border:1px solid #e2e8f0; border-left:4px solid #3b82f6;
                          border-radius:4px 16px 16px 16px; padding:14px 16px;
                          max-width:95%; font-size:0.95rem; color:#1e293b;
                          box-shadow:0 2px 8px rgba(0,0,0,0.05);">
                <b>🤖 Drive Wise</b> <span style="font-size:0.75rem; color:#64748b;">({item['car']} · {item['time']}s)</span><br><br>
                {answer_html}
              </div>
            </div>
            """))
            # Sources
            if item.get('sources'):
                src_html = "".join([
                    f"<span style='display:inline-block; background:#f0fdf4; color:#166534; "
                    f"font-size:0.75rem; padding:2px 8px; border-radius:20px; margin:2px; "
                    f"border:1px solid #bbf7d0;'>"
                    f"[{i+1}] Page {s['page']} · {s['section']}</span>"
                    for i, s in enumerate(item['sources'])
                ])
                display(HTML(f"""
                <div style="margin:-8px 0 12px 12px; font-size:0.8rem; color:#64748b;">
                  📚 Sources: {src_html}
                </div>
                """))
            display(HTML("<hr style='border:none; border-top:1px solid #f1f5f9; margin:4px 0;'>"))

# ── Ask button click handler ────────────────────────────────────────────────────
def on_ask_clicked(b):
    query = query_input.value.strip()
    if not query:
        status_label.value = '<span style="color:#ef4444;">⚠️ Please type a question first!</span>'
        return

    brand = brand_dropdown.value
    model = model_dropdown.value

    ask_button.disabled  = True
    clear_button.disabled = True
    ask_button.description = ' Thinking...'
    status_label.value = f'<span style="color:#3b82f6;">⏳ Searching brochure and generating answer for {brand} {model}...</span>'

    # Clear input
    query_input.value = ''

    # Run RAG
    start = time.time()
    answer, sources = generate_answer(query, brand, model)
    elapsed = round(time.time() - start, 2)

    # Append to history
    chat_history.append({
        'query': query,
        'answer': answer,
        'sources': sources,
        'car': f'{brand} {model}',
        'time': elapsed
    })

    render_chat()

    status_label.value = f'<span style="color:#10b981;">✅ Answered in {elapsed}s · {len(sources)} brochure chunk(s) used</span>'
    ask_button.disabled  = False
    clear_button.disabled = False
    ask_button.description = ' Ask DriveWise'

# ── Clear button handler ────────────────────────────────────────────────────────
def on_clear_clicked(b):
    chat_history.clear()
    render_chat()
    status_label.value = '<span style="color:#64748b;">🗑️ Chat cleared. Ready for new questions!</span>'

ask_button.on_click(on_ask_clicked)
clear_button.on_click(on_clear_clicked)

# ── Assemble the UI ─────────────────────────────────────────────────────────────
header = HTML("""
<div style="background:linear-gradient(135deg,#1d4ed8,#3b82f6); color:white;
             padding:16px 20px; border-radius:12px 12px 0 0; margin-bottom:0;">
  <span style="font-size:1.3rem; font-weight:700;">🚗 Drive Wise — Interactive Chat</span>
  <span style="font-size:0.85rem; opacity:0.85; margin-left:12px;">Ask anything about the car brochures</span>
</div>
""")

car_selector = widgets.HBox(
    [brand_dropdown, model_dropdown],
    layout=widgets.Layout(gap='12px', margin='8px 0')
)

button_row = widgets.HBox(
    [ask_button, clear_button, status_label],
    layout=widgets.Layout(gap='10px', align_items='center', margin='8px 0')
)

query_label = HTML('<div style="font-weight:600; color:#334155; margin:4px 0 2px 0;">💬 Your Question:</div>')

ui = widgets.VBox(
    [header, car_selector, query_label, query_input, button_row, chat_output],
    layout=widgets.Layout(
        border='1px solid #e2e8f0',
        border_radius='12px',
        padding='16px',
        width='100%',
        background_color='#ffffff'
    )
)

render_chat()
display(ui)

---
## 📊 Quality Evaluation (Optional)

Run this cell after chatting to get an LLM-as-a-Judge score for your last question.

In [ ]:
# Evaluate quality of the last question in the chat
if not chat_history:
    print("⚠️  No questions asked yet. Use the chat above first!")
else:
    last = chat_history[-1]
    print(f"Evaluating: '{last['query']}'")
    ctx = "".join([f"[{i+1}] {s['text'][:200]}\n" for i, s in enumerate(last['sources'])])

    eval_prompt = f"""
You are a RAG quality judge. Evaluate this output:
Query: "{last['query']}"
Context: {ctx}
Answer: "{last['answer']}"

Score 1.0–5.0 for each. Respond ONLY with valid JSON:
{{"faithfulness": float, "context_relevance": float, "answer_correctness": float, "rationale": "brief explanation"}}
"""
    eval_model = genai.GenerativeModel("models/gemini-2.5-flash")
    resp = eval_model.generate_content(eval_prompt)
    text = resp.text.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    scores = json.loads(text)

    display(HTML(f"""
    <div style="border:1px solid #e2e8f0; border-radius:12px; padding:16px; background:#f8fafc; margin-top:12px;">
      <h3 style="margin:0 0 12px 0; color:#1e293b;">📊 Quality Evaluation Results</h3>
      <div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:12px;">
        <div style="background:white; border:1px solid #ddd; border-radius:8px; padding:12px 20px; text-align:center; min-width:120px;">
          <div style="font-size:1.8rem; font-weight:700; color:#3b82f6;">{scores.get('faithfulness',0):.1f}</div>
          <div style="font-size:0.8rem; color:#64748b; text-transform:uppercase;">Faithfulness</div>
        </div>
        <div style="background:white; border:1px solid #ddd; border-radius:8px; padding:12px 20px; text-align:center; min-width:120px;">
          <div style="font-size:1.8rem; font-weight:700; color:#10b981;">{scores.get('context_relevance',0):.1f}</div>
          <div style="font-size:0.8rem; color:#64748b; text-transform:uppercase;">Ctx Relevance</div>
        </div>
        <div style="background:white; border:1px solid #ddd; border-radius:8px; padding:12px 20px; text-align:center; min-width:120px;">
          <div style="font-size:1.8rem; font-weight:700; color:#8b5cf6;">{scores.get('answer_correctness',0):.1f}</div>
          <div style="font-size:0.8rem; color:#64748b; text-transform:uppercase;">Correctness</div>
        </div>
      </div>
      <div style="font-size:0.9rem; color:#475569; background:white; padding:10px; border-radius:8px; border:1px solid #e2e8f0;">
        📝 <b>Rationale:</b> {scores.get('rationale','N/A')}
      </div>
    </div>
    """))

---
<div style="background:#f0fdf4; border:1px solid #86efac; border-radius:12px; padding:1.5rem; text-align:center;">
  <h3 style="color:#166534; margin:0;">✅ DriveWise Demo Complete!</h3>
  <p style="color:#15803d; margin:0.5rem 0 0 0;">
    Full pipeline: PDF indexing → Gemini embeddings → hybrid retrieval → grounded generation → LLM evaluation
  </p>
</div>

### 📌 Design Decisions
| Component | Choice | Reason |
|-----------|--------|---------|
| Embeddings | `gemini-embedding-001` | High-quality multilingual embeddings |
| Generation | `gemini-2.5-flash` | Fast, cost-efficient, strong instruction following |
| Retrieval | Hybrid (80% semantic + 20% keyword) | Captures meaning AND exact spec numbers |
| Chunking | Paragraph-level (~800 chars) | Balances context and precision |
| Section Tags | Rule-based keyword scoring | Metadata pre-filter reduces noise |
| Evaluation | LLM-as-a-Judge | Scalable quality measurement without ground truth |